# BertDiffused — Inference & Demo

Interactive showcase of **BertMoEDiffusion** — a BERT-base masked diffusion
language model enhanced with Mixture-of-Experts and LoRA adapters.

### What this notebook demonstrates

| Demo | Description |
|------|-------------|
| **Unconditional Generation** | Generate text from a fully masked sequence |
| **Text Infilling** | Fill in a missing span given prefix + suffix context |
| **Keyword-Constrained Generation** | Generate fluent text containing required keywords |
| **Denoising Visualisation** | Watch the reverse diffusion process unfold step-by-step |
| **MoE Expert Analysis** | Inspect which experts activate at different noise levels |
| **Diffusion Steps vs Quality** | See how generation quality scales with compute |

## 0. Setup

In [ ]:
import os, sys, torch, math, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from IPython.display import display, HTML, Markdown
from pathlib import Path

# If running on Colab, mount Drive and clone repo
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists('/content/BertDiffused'):
        !git clone https://github.com/stephenlee/BertDiffused.git /content/BertDiffused
    os.chdir('/content/BertDiffused')
    !pip install -q -r requirements.txt

# Ensure repo root is on path
REPO_ROOT = Path('.').resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB)")

## 1. Load Model

In [ ]:
import yaml
from transformers import AutoTokenizer
from model import BertMoEDiffusion, LogLinearNoiseSchedule

# ── Config ────────────────────────────────────────────────────────────────
with open('configs/config.yaml') as f:
    cfg = yaml.safe_load(f)

moe_cfg = cfg['model']['moe']
lora_cfg = cfg['model']['lora']

# ── Tokenizer ─────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(cfg['model']['backbone'])

# ── Model ─────────────────────────────────────────────────────────────────
model = BertMoEDiffusion(
    bert_model_name=cfg['model']['backbone'],
    moe_layers=moe_cfg['moe_layers'],
    num_experts=moe_cfg['num_experts'],
    num_experts_per_token=moe_cfg['num_experts_per_token'],
    expert_hidden_multiplier=moe_cfg['expert_hidden_multiplier'],
    router_jitter=0.0,  # no jitter at inference
    router_z_loss_coef=moe_cfg['router_z_loss_coef'],
    router_aux_loss_coef=moe_cfg['router_aux_loss_coef'],
    time_embed_dim=cfg['model']['time_embed_dim'],
    use_time_conditioning=cfg['model']['use_time_conditioning'],
    dropout=0.0,  # no dropout at inference
    lora_enabled=False,  # assume merged weights
)
model.set_mask_token_id(tokenizer.mask_token_id)

# ── Load trained weights ──────────────────────────────────────────────────
# Update this path to your trained checkpoint
CKPT_PATH = 'checkpoints/bertdiffused/final_model.pt'  # local
# CKPT_PATH = '/content/drive/MyDrive/BertDiffused/final_model/final_model.pt'  # Colab

if os.path.exists(CKPT_PATH):
    ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
    state = ckpt.get('model', ckpt)  # handle both formats
    model.load_state_dict(state, strict=False)
    print(f"Loaded checkpoint: {CKPT_PATH}")
else:
    print(f"WARNING: No checkpoint found at {CKPT_PATH}")
    print("Running with pretrained BERT weights only (results will be noisy).")

model.to(device)
model.eval()

noise_schedule = LogLinearNoiseSchedule()

ps = model.trainable_parameters_summary()
print(f"\nModel: {ps['total']:,} params")
print(f"MoE experts: {moe_cfg['num_experts']} per layer, top-{moe_cfg['num_experts_per_token']} routing")
print(f"MoE layers: {moe_cfg['moe_layers']}")

---
## 2. Unconditional Generation

Start from a fully `[MASK]`ed sequence and let the diffusion model denoise it into coherent text.

In [ ]:
@torch.no_grad()
def generate_unconditional(model, tokenizer, noise_schedule, device,
                           seq_len=64, num_steps=200, num_samples=5):
    """Generate text from a fully masked starting point."""
    generated = model.sample(
        batch_size=num_samples,
        seq_len=seq_len,
        num_steps=num_steps,
        noise_schedule=noise_schedule,
        tokenizer=tokenizer,
        device=device,
    )
    texts = []
    for i in range(num_samples):
        text = tokenizer.decode(generated[i], skip_special_tokens=True,
                                clean_up_tokenization_spaces=True)
        texts.append(text)
    return texts

print("Generating unconditional samples (T=200 steps)...\n")
samples = generate_unconditional(model, tokenizer, noise_schedule, device)

for i, text in enumerate(samples, 1):
    display(Markdown(f"**Sample {i}:** {text}"))

---
## 3. Text Infilling

The model's **bidirectional attention** allows it to use *both* the prefix and suffix when filling in a missing span — something autoregressive models can't do.

In [ ]:
@torch.no_grad()
def infill(model, tokenizer, noise_schedule, device,
           prefix, suffix, span_len=15, num_steps=200, num_samples=3):
    """Fill in a gap between prefix and suffix."""
    prefix_ids = tokenizer.encode(prefix, add_special_tokens=False)
    suffix_ids = tokenizer.encode(suffix, add_special_tokens=False)
    seq_len = len(prefix_ids) + span_len + len(suffix_ids)

    prefix_t = torch.tensor([prefix_ids] * num_samples, dtype=torch.long, device=device)
    suffix_t = torch.tensor([suffix_ids] * num_samples, dtype=torch.long, device=device)

    mask_id = tokenizer.mask_token_id
    z = torch.full((num_samples, seq_len), mask_id, dtype=torch.long, device=device)
    fixed = torch.zeros(num_samples, seq_len, dtype=torch.bool, device=device)

    # Pin prefix
    z[:, :len(prefix_ids)] = prefix_t
    fixed[:, :len(prefix_ids)] = True
    # Pin suffix
    z[:, len(prefix_ids) + span_len:] = suffix_t
    fixed[:, len(prefix_ids) + span_len:] = True

    generated = model.sample(
        batch_size=num_samples, seq_len=seq_len, num_steps=num_steps,
        noise_schedule=noise_schedule, tokenizer=tokenizer, device=device,
        fixed_token_mask=fixed,
    )

    results = []
    for i in range(num_samples):
        full = tokenizer.decode(generated[i], skip_special_tokens=True,
                                clean_up_tokenization_spaces=True)
        span = tokenizer.decode(generated[i, len(prefix_ids):len(prefix_ids)+span_len],
                                skip_special_tokens=True,
                                clean_up_tokenization_spaces=True)
        results.append({'full': full, 'span': span})
    return results

In [ ]:
infill_examples = [
    {"prefix": "The scientist carefully examined the",
     "suffix": "and published the results in a leading journal."},
    {"prefix": "After years of training, the athlete",
     "suffix": "breaking the previous world record."},
    {"prefix": "The old library on the corner of the street",
     "suffix": "and the children loved spending time there."},
]

for ex in infill_examples:
    display(Markdown(f"---\n**Prefix:** *{ex['prefix']}*\n\n**Suffix:** *{ex['suffix']}*"))
    results = infill(model, tokenizer, noise_schedule, device,
                     prefix=ex['prefix'], suffix=ex['suffix'], num_steps=200)
    for j, r in enumerate(results, 1):
        display(Markdown(f"  {j}. **[{r['span']}]** → {r['full']}"))

---
## 4. Keyword-Constrained Generation

Pin specific keywords at fixed positions in the sequence. The diffusion model fills in the rest while **guaranteeing keyword satisfaction** — an inherent advantage over left-to-right generation.

In [ ]:
@torch.no_grad()
def constrained_generate(model, tokenizer, noise_schedule, device,
                         keywords, seq_len=48, num_steps=200, num_samples=3):
    """Generate text that must contain the given keywords."""
    mask_id = tokenizer.mask_token_id
    z = torch.full((num_samples, seq_len), mask_id, dtype=torch.long, device=device)
    fixed = torch.zeros(num_samples, seq_len, dtype=torch.bool, device=device)

    # Place keywords at evenly spaced positions
    positions = np.linspace(4, seq_len - 4, len(keywords), dtype=int)
    kw_info = []
    for kw, pos in zip(keywords, positions):
        kw_ids = tokenizer.encode(' ' + kw, add_special_tokens=False)
        for offset, kid in enumerate(kw_ids):
            if pos + offset < seq_len:
                z[:, pos + offset] = kid
                fixed[:, pos + offset] = True
        kw_info.append((kw, int(pos)))

    generated = model.sample(
        batch_size=num_samples, seq_len=seq_len, num_steps=num_steps,
        noise_schedule=noise_schedule, tokenizer=tokenizer, device=device,
        fixed_token_mask=fixed,
    )

    results = []
    for i in range(num_samples):
        text = tokenizer.decode(generated[i], skip_special_tokens=True,
                                clean_up_tokenization_spaces=True)
        # Highlight keywords in the output
        highlighted = text
        for kw in keywords:
            highlighted = highlighted.replace(kw, f'**{kw}**')
        satisfied = all(kw.lower() in text.lower() for kw in keywords)
        results.append({'text': text, 'highlighted': highlighted, 'satisfied': satisfied})
    return results, kw_info

In [ ]:
keyword_sets = [
    ["ocean", "discovered", "ancient"],
    ["music", "played", "night"],
    ["engineer", "designed", "bridge"],
    ["rain", "fell", "evening"],
    ["children", "laughed", "park"],
]

total_satisfied = 0
total_generated = 0

for keywords in keyword_sets:
    display(Markdown(f"---\n**Keywords:** `{'`, `'.join(keywords)}`"))
    results, kw_info = constrained_generate(
        model, tokenizer, noise_schedule, device, keywords=keywords, num_steps=200
    )
    for j, r in enumerate(results, 1):
        status = "OK" if r['satisfied'] else "MISS"
        display(Markdown(f"  {j}. [{status}] {r['highlighted']}"))
        total_satisfied += int(r['satisfied'])
        total_generated += 1

print(f"\nKeyword satisfaction: {total_satisfied}/{total_generated} ({100*total_satisfied/total_generated:.0f}%)")

---
## 5. Denoising Visualisation

Watch a fully masked sequence transform into coherent text. Tokens are colour-coded:
- <span style="color:red">[MASK]</span> — still masked
- <span style="color:green">token</span> — just unmasked at this step
- <span style="color:black">token</span> — was unmasked in a prior step

In [ ]:
@torch.no_grad()
def generate_with_trajectory(model, tokenizer, noise_schedule, device,
                             seq_len=48, num_steps=100, snapshot_steps=None):
    """Generate one sequence and capture intermediate states."""
    mask_id = tokenizer.mask_token_id
    model.eval()
    model.mask_token_id = mask_id

    z = torch.full((1, seq_len), mask_id, dtype=torch.long, device=device)
    timesteps = torch.linspace(1.0, 0.0, num_steps + 1, device=device)

    if snapshot_steps is None:
        # Capture ~10 snapshots evenly spaced
        snapshot_steps = set(int(i) for i in np.linspace(0, num_steps - 1, 10, dtype=int))
    else:
        snapshot_steps = set(snapshot_steps)

    trajectory = []

    for i in range(num_steps):
        t_val = timesteps[i]
        s_val = timesteps[i + 1]
        t_batch = torch.full((1,), t_val, device=device)
        s_batch = torch.full((1,), s_val, device=device)

        logits = model.forward(z, t_batch)
        posterior = noise_schedule.posterior_logits(
            logits_x0=logits, z_t=z, t=t_batch, s=s_batch, mask_token_id=mask_id,
        )
        z_new = torch.distributions.Categorical(logits=posterior).sample()
        carry = (z != mask_id)
        z_new[carry] = z[carry]

        if i in snapshot_steps:
            trajectory.append({
                'step': i,
                't': t_val.item(),
                'ids': z_new[0].clone(),
                'prev_ids': z[0].clone(),
            })

        z = z_new

    # Final state
    trajectory.append({
        'step': num_steps,
        't': 0.0,
        'ids': z[0].clone(),
        'prev_ids': z[0].clone(),
    })

    return trajectory


def render_trajectory(trajectory, tokenizer, mask_id):
    """Render the denoising trajectory as coloured HTML."""
    html_parts = ['<div style="font-family: monospace; font-size: 13px; line-height: 2.0;">']
    for snap in trajectory:
        step = snap['step']
        t = snap['t']
        ids = snap['ids']
        prev = snap['prev_ids']

        tokens_html = []
        for j, (tid, pid) in enumerate(zip(ids, prev)):
            tid_val = tid.item()
            pid_val = pid.item()
            if tid_val == mask_id:
                tokens_html.append('<span style="color:#d32f2f;">[M]</span>')
            elif pid_val == mask_id:
                word = tokenizer.decode([tid_val])
                tokens_html.append(f'<span style="color:#2e7d32; font-weight:bold;">{word}</span>')
            else:
                word = tokenizer.decode([tid_val])
                tokens_html.append(f'<span style="color:#333;">{word}</span>')

        line = ' '.join(tokens_html)
        html_parts.append(f'<div><b>Step {step:3d}</b> (t={t:.2f}): {line}</div>')

    html_parts.append('</div>')
    return '\n'.join(html_parts)


print("Generating with trajectory capture (T=100)...")
trajectory = generate_with_trajectory(model, tokenizer, noise_schedule, device,
                                      seq_len=48, num_steps=100)

display(HTML(render_trajectory(trajectory, tokenizer, tokenizer.mask_token_id)))

# Print final text
final_text = tokenizer.decode(trajectory[-1]['ids'], skip_special_tokens=True,
                               clean_up_tokenization_spaces=True)
display(Markdown(f"\n**Final:** {final_text}"))

---
## 6. Unmasking Progress Curve

Plot the fraction of tokens unmasked at each diffusion step. The log-linear schedule $\alpha(t) = 1 - t$ means masking probability $= t$, so unmasking accelerates toward $t=0$.

In [ ]:
mask_id = tokenizer.mask_token_id

steps = [snap['step'] for snap in trajectory]
pct_unmasked = [(snap['ids'] != mask_id).float().mean().item() * 100 for snap in trajectory]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(steps, pct_unmasked, 'b-o', markersize=5, linewidth=1.5)
ax.set_xlabel('Diffusion Step', fontsize=12)
ax.set_ylabel('Tokens Unmasked (%)', fontsize=12)
ax.set_title('Reverse Diffusion: Unmasking Progress', fontsize=13)
ax.set_ylim(-5, 105)
ax.grid(True, alpha=0.3)
ax.axhline(100, color='green', linestyle='--', alpha=0.4, label='Fully unmasked')
ax.legend()
plt.tight_layout()
plt.show()

---
## 7. Diffusion Steps vs Quality

Trade off generation quality against compute: fewer steps = faster but noisier.

In [ ]:
step_counts = [10, 25, 50, 100, 200, 500]
step_samples = {}

print("Generating at different step counts:")
for T in step_counts:
    t0 = time.time()
    texts = generate_unconditional(model, tokenizer, noise_schedule, device,
                                   seq_len=48, num_steps=T, num_samples=3)
    elapsed = time.time() - t0
    step_samples[T] = texts
    print(f"  T={T:4d} — {elapsed:.2f}s")

# Display results side by side
for T in step_counts:
    display(Markdown(f"---\n### T = {T} steps"))
    for j, text in enumerate(step_samples[T], 1):
        display(Markdown(f"  {j}. {text}"))

---
## 8. MoE Expert Routing Analysis

Inspect how the MoE router distributes tokens to experts at different noise levels $t$. We expect different experts to specialize:
- **High $t$** (heavily masked): global language modelling experts
- **Low $t$** (mostly unmasked): local refinement experts

In [ ]:
@torch.no_grad()
def get_expert_routing(model, tokenizer, noise_schedule, device,
                       t_values=None, seq_len=64, batch_size=8):
    """Collect expert routing distributions at different noise levels."""
    if t_values is None:
        t_values = [0.1, 0.3, 0.5, 0.7, 0.9]

    mask_id = tokenizer.mask_token_id
    model.eval()

    # Generate a batch of "clean" token sequences as dummy input
    dummy_text = "The quick brown fox jumps over the lazy dog and runs across the field " * 10
    enc = tokenizer(dummy_text, return_tensors='pt', max_length=seq_len,
                    truncation=True, padding='max_length')
    x_clean = enc['input_ids'].repeat(batch_size, 1).to(device)

    results = {}  # t_val -> {layer_idx -> routing_counts (num_experts,)}

    for t_val in t_values:
        t = torch.full((batch_size,), t_val, device=device)
        z_t = noise_schedule.noise_sequence(x_clean, t, mask_id)

        # Forward pass to trigger routing
        _ = model.forward(z_t, t)

        routing = {}
        for layer_idx, moe_ffn in model._moe_layer_map.items():
            if hasattr(moe_ffn, 'router') and hasattr(moe_ffn.router, '_last_expert_counts'):
                counts = moe_ffn.router._last_expert_counts
                if counts is not None:
                    routing[layer_idx] = counts.float().cpu().numpy()
            elif hasattr(moe_ffn, '_last_routing_weights'):
                weights = moe_ffn._last_routing_weights
                if weights is not None:
                    routing[layer_idx] = weights.float().mean(0).cpu().numpy()

        results[t_val] = routing

    return results


def plot_expert_heatmaps(routing_data, moe_layers, num_experts):
    """Plot expert utilisation heatmaps."""
    t_values = sorted(routing_data.keys())

    # Build a matrix: rows = t values, cols = (layer, expert) pairs
    col_labels = []
    for l in moe_layers:
        for e in range(num_experts):
            col_labels.append(f'L{l}-E{e}')

    matrix = np.zeros((len(t_values), len(col_labels)))
    for i, t_val in enumerate(t_values):
        col = 0
        for l in moe_layers:
            if l in routing_data[t_val]:
                counts = routing_data[t_val][l]
                n = min(len(counts), num_experts)
                matrix[i, col:col + n] = counts[:n]
            col += num_experts

    # Normalise rows
    row_sums = matrix.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    matrix = matrix / row_sums

    fig, ax = plt.subplots(figsize=(16, 5))
    im = ax.imshow(matrix, aspect='auto', cmap='YlOrRd', interpolation='nearest')
    ax.set_yticks(range(len(t_values)))
    ax.set_yticklabels([f't={t:.1f}' for t in t_values])
    ax.set_xlabel('Layer-Expert', fontsize=11)
    ax.set_ylabel('Noise Level t', fontsize=11)
    ax.set_title('MoE Expert Routing Distribution by Noise Level', fontsize=13)

    # Add layer separators
    for i in range(1, len(moe_layers)):
        ax.axvline(i * num_experts - 0.5, color='white', linewidth=2)

    plt.colorbar(im, ax=ax, label='Routing Weight (normalised)', shrink=0.8)
    plt.tight_layout()
    plt.show()


print("Collecting expert routing data...")
routing_data = get_expert_routing(model, tokenizer, noise_schedule, device)

if any(len(v) > 0 for v in routing_data.values()):
    plot_expert_heatmaps(routing_data, moe_cfg['moe_layers'], moe_cfg['num_experts'])
else:
    print("Note: Expert routing counts not exposed by this MoE implementation.")
    print("Routing analysis requires the router to store `_last_expert_counts` during forward.")

---
## 9. Interactive Generation

Try your own prompts below.

In [ ]:
# ── Unconditional ────────────────────────────────────────────────────────
# Change seq_len and num_steps to explore
my_samples = generate_unconditional(
    model, tokenizer, noise_schedule, device,
    seq_len=64, num_steps=200, num_samples=3,
)
for i, text in enumerate(my_samples, 1):
    display(Markdown(f"**{i}.** {text}"))

In [ ]:
# ── Infilling ─────────────────────────────────────────────────────────────
# Edit the prefix and suffix to try your own infilling examples
my_prefix = "In a breakthrough experiment, researchers at MIT"
my_suffix = "which could revolutionize renewable energy."

results = infill(model, tokenizer, noise_schedule, device,
                 prefix=my_prefix, suffix=my_suffix,
                 span_len=12, num_steps=200, num_samples=5)

display(Markdown(f"**Prefix:** *{my_prefix}*\n\n**Suffix:** *{my_suffix}*\n"))
for j, r in enumerate(results, 1):
    display(Markdown(f"  {j}. [{r['span']}] → {r['full']}"))

In [ ]:
# ── Keyword-constrained ───────────────────────────────────────────────────
# Edit the keywords list
my_keywords = ["quantum", "computer", "solved"]

results, kw_info = constrained_generate(
    model, tokenizer, noise_schedule, device,
    keywords=my_keywords, seq_len=48, num_steps=200, num_samples=5,
)

display(Markdown(f"**Keywords:** `{'`, `'.join(my_keywords)}`\n"))
for j, r in enumerate(results, 1):
    status = "OK" if r['satisfied'] else "MISS"
    display(Markdown(f"  {j}. [{status}] {r['highlighted']}"))

---

### Summary

| Capability | How |
|---|---|
| **Unconditional generation** | All `[MASK]` → reverse diffuse → text |
| **Text infilling** | Pin prefix+suffix, mask the gap, diffuse the span |
| **Keyword constraints** | Pin keyword tokens at fixed positions, diffuse the rest |
| **Quality scaling** | More diffusion steps = higher quality (diminishing returns past ~200) |
| **MoE specialisation** | Different experts activate for different noise levels |

For quantitative benchmarks (BLEU-4, MAUVE, PPL), run the evaluation scripts:
```bash
python -m tasks.infilling --model_path checkpoints/bertdiffused/final_model.pt
python -m tasks.constrained_gen --model_path checkpoints/bertdiffused/final_model.pt
python -m eval.compare --diffusion_ckpt checkpoints/bertdiffused
```